In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

print("Libraries imported successfully.")

In [ ]:
DATA_PATH = "../data/processed/final_features.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Shape: (10000, 33)


,year,state,district,season,crop_type,seed_variety,area_sown_hectares,irrigation_type,rainfall_mm,temperature_min_c,...,yield_tonnes_per_hectare,sowing_year,sowing_month,sowing_day,sowing_day_of_year,temperature_range_c,rainfall_per_day,total_npk_kg_ha,np_ratio,kn_ratio
0,2022,Karnataka,Belagavi,Rabi,Maize,HQPM-1,1.52,Rainfed,573.7,17.5,...,2.82,2022,11,10,314,19.5,5.794949,221.0,1.541985,1.086634
1,2019,Karnataka,Kalaburagi,Rabi,Maize,HQPM-1,22.19,Rainfed,805.9,17.3,...,2.33,2019,11,25,329,14.2,7.393578,300.3,2.497600,0.523382
2,2019,Punjab,Amritsar,Rabi,Wheat,HD-2967,8.50,Drip Irrigated,343.2,7.5,...,2.31,2019,11,15,319,18.7,2.908475,139.8,1.144231,0.475630
3,2017,Madhya Pradesh,Jabalpur,Rabi,Wheat,HD-3086,7.48,Irrigated,339.7,14.9,...,1.72,2017,10,29,302,12.3,2.342759,227.1,1.935323,0.429306
4,2022,Rajasthan,Bikaner,Kharif,Cotton,Bunny-Bt,6.44,Rainfed,564.1,20.3,...,1.46,2022,6,4,155,13.5,3.016578,200.6,5.274900,0.325529


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   year                      10000 non-null  int64  
 1   state                     10000 non-null  str    
 2   district                  10000 non-null  str    
 3   season                    10000 non-null  str    
 4   crop_type                 10000 non-null  str    
 5   seed_variety              10000 non-null  str    
 6   area_sown_hectares        10000 non-null  float64
 7   irrigation_type           10000 non-null  str    
 8   rainfall_mm               10000 non-null  float64
 9   temperature_min_c         10000 non-null  float64
 10  temperature_max_c         10000 non-null  float64
 11  temperature_avg_c         10000 non-null  float64
 12  humidity_pct              10000 non-null  float64
 13  growing_season_days       10000 non-null  int64  
 14  soil_ph           

In [ ]:
print("\nData types:")
display(df.dtypes)

Missing values:
 Series([], dtype: int64)

Duplicate rows: 0


In [ ]:
print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

All columns:
 ['year', 'state', 'district', 'season', 'crop_type', 'seed_variety', 'area_sown_hectares', 'irrigation_type', 'rainfall_mm', 'temperature_min_c', 'temperature_max_c', 'temperature_avg_c', 'humidity_pct', 'growing_season_days', 'soil_ph', 'nitrogen_kg_ha', 'phosphorus_kg_ha', 'potassium_kg_ha', 'soil_type', 'soil_moisture_pct', 'previous_yield_tonnes_ha', 'yield_trend_pct_yoy', 'ndvi', 'yield_tonnes_per_hectare', 'sowing_year', 'sowing_month', 'sowing_day', 'sowing_day_of_year', 'temperature_range_c', 'rainfall_per_day', 'total_npk_kg_ha', 'np_ratio', 'kn_ratio']


In [ ]:
TARGET = "yield_tonnes_per_hectare"

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found in dataset.")

print("Target column:", TARGET)

Target: yield_tonnes_per_hectare

Drop columns: ['record_id']

Numerical features: ['year', 'area_sown_hectares', 'rainfall_mm', 'temperature_min_c', 'temperature_max_c', 'temperature_avg_c', 'humidity_pct', 'growing_season_days', 'soil_ph', 'nitrogen_kg_ha', 'phosphorus_kg_ha', 'potassium_kg_ha', 'soil_moisture_pct', 'previous_yield_tonnes_ha', 'yield_trend_pct_yoy', 'ndvi', 'sowing_year', 'sowing_month', 'sowing_day', 'sowing_day_of_year', 'temperature_range_c', 'rainfall_per_day', 'total_npk_kg_ha', 'np_ratio', 'kn_ratio']

Categorical features: ['state', 'district', 'season', 'crop_type', 'seed_variety', 'irrigation_type', 'soil_type']


C:\Users\HP\AppData\Local\Temp\ipykernel_16476\1585212123.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df[feature_cols].select_dtypes(include=["object"]).columns.tolist()


In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

Correlation with target:
previous_yield_tonnes_ha    0.773783
ndvi                        0.193981
yield_tonnes_per_hectare    1.000000
Name: yield_tonnes_per_hectare, dtype: float64


In [ ]:
print(y.describe())

X shape: (10000, 32)
y shape: (10000,)


,year,state,district,season,crop_type,seed_variety,area_sown_hectares,irrigation_type,rainfall_mm,temperature_min_c,...,ndvi,sowing_year,sowing_month,sowing_day,sowing_day_of_year,temperature_range_c,rainfall_per_day,total_npk_kg_ha,np_ratio,kn_ratio
0,2022,Karnataka,Belagavi,Rabi,Maize,HQPM-1,1.52,Rainfed,573.7,17.5,...,0.703,2022,11,10,314,19.5,5.794949,221.0,1.541985,1.086634
1,2019,Karnataka,Kalaburagi,Rabi,Maize,HQPM-1,22.19,Rainfed,805.9,17.3,...,0.707,2019,11,25,329,14.2,7.393578,300.3,2.497600,0.523382
2,2019,Punjab,Amritsar,Rabi,Wheat,HD-2967,8.50,Drip Irrigated,343.2,7.5,...,0.649,2019,11,15,319,18.7,2.908475,139.8,1.144231,0.475630
3,2017,Madhya Pradesh,Jabalpur,Rabi,Wheat,HD-3086,7.48,Irrigated,339.7,14.9,...,0.575,2017,10,29,302,12.3,2.342759,227.1,1.935323,0.429306
4,2022,Rajasthan,Bikaner,Kharif,Cotton,Bunny-Bt,6.44,Rainfed,564.1,20.3,...,0.651,2022,6,4,155,13.5,3.016578,200.6,5.274900,0.325529


In [ ]:
possible_id_columns = [
    col for col in X.columns
    if "id" in col.lower()
]

print("Possible ID columns:", possible_id_columns)

X_train: (8000, 32)
X_test: (2000, 32)
y_train: (8000,)
y_test: (2000,)


In [ ]:
if "record_id" in X.columns:
    X = X.drop(columns=["record_id"])
    print("Removed record_id.")

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created.")

In [ ]:
OneHotEncoder(
    handle_unknown="ignore",
    sparse=False
)

In [ ]:
linear_regression_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

In [ ]:
linear_regression_pipeline.fit(X_train, y_train)

print("Linear Regression trained successfully.")

In [ ]:
y_pred_lr = linear_regression_pipeline.predict(X_test)

In [ ]:
lr_mae = mean_absolute_error(y_test, y_pred_lr)

lr_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_lr)
)

lr_r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression")
print("-------------------------")
print(f"MAE  : {lr_mae:.4f}")
print(f"RMSE : {lr_rmse:.4f}")
print(f"R²   : {lr_r2:.4f}")

In [ ]:
decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeRegressor(
                random_state=42,
                max_depth=None
            )
        )
    ]
)

In [ ]:
decision_tree_pipeline.fit(X_train, y_train)

print("Decision Tree trained successfully.")

In [ ]:
y_pred_dt = decision_tree_pipeline.predict(X_test)

In [ ]:
dt_mae = mean_absolute_error(y_test, y_pred_dt)

dt_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_dt)
)

dt_r2 = r2_score(y_test, y_pred_dt)

print("Decision Tree")
print("-------------------------")
print(f"MAE  : {dt_mae:.4f}")
print(f"RMSE : {dt_rmse:.4f}")
print(f"R²   : {dt_r2:.4f}")

In [ ]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                max_features="sqrt"
            )
        )
    ]
)

In [ ]:
random_forest_pipeline.fit(X_train, y_train)

print("Random Forest trained successfully.")

In [ ]:
y_pred_rf = random_forest_pipeline.predict(X_test)

In [ ]:
rf_mae = mean_absolute_error(y_test, y_pred_rf)

rf_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_rf)
)

rf_r2 = r2_score(y_test, y_pred_rf)

print("Random Forest")
print("-------------------------")
print(f"MAE  : {rf_mae:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"R²   : {rf_r2:.4f}")

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "MAE": [
        lr_mae,
        dt_mae,
        rf_mae
    ],
    "RMSE": [
        lr_rmse,
        dt_rmse,
        rf_rmse
    ],
    "R2": [
        lr_r2,
        dt_r2,
        rf_r2
    ]
})

results

In [ ]:
display(
    results.style.format({
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "R2": "{:.4f}"
    })
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=results,
    x="Model",
    y="RMSE"
)

plt.title("Model Comparison - RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=results,
    x="Model",
    y="R2"
)

plt.title("Model Comparison - R²")
plt.xlabel("Model")
plt.ylabel("R² Score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y_test,
    y_pred_lr,
    alpha=0.6
)

min_value = min(y_test.min(), y_pred_lr.min())
max_value = max(y_test.max(), y_pred_lr.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Yield")
plt.ylabel("Predicted Yield")
plt.title("Linear Regression - Actual vs Predicted")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y_test,
    y_pred_dt,
    alpha=0.6
)

min_value = min(y_test.min(), y_pred_dt.min())
max_value = max(y_test.max(), y_pred_dt.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Yield")
plt.ylabel("Predicted Yield")
plt.title("Decision Tree - Actual vs Predicted")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y_test,
    y_pred_rf,
    alpha=0.6
)

min_value = min(y_test.min(), y_pred_rf.min())
max_value = max(y_test.max(), y_pred_rf.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Yield")
plt.ylabel("Predicted Yield")
plt.title("Random Forest - Actual vs Predicted")

plt.tight_layout()
plt.show()

In [ ]:
rf_preprocessor = random_forest_pipeline.named_steps["preprocessor"]

feature_names = rf_preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))

In [ ]:
rf_model = random_forest_pipeline.named_steps["model"]

importances = rf_model.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

display(feature_importance.head(20))

In [ ]:
top_features = feature_importance.head(15).sort_values(
    by="Importance"
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 15 Random Forest Feature Importances")

plt.tight_layout()
plt.show()

In [ ]:
MODEL_DIR = Path("../models/member1")

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Model directory:", MODEL_DIR)

In [ ]:
joblib.dump(
    linear_regression_pipeline,
    MODEL_DIR / "linear_regression.pkl"
)

joblib.dump(
    decision_tree_pipeline,
    MODEL_DIR / "decision_tree.pkl"
)

joblib.dump(
    random_forest_pipeline,
    MODEL_DIR / "random_forest.pkl"
)

print("All Member 1 models saved successfully.")

In [ ]:
RESULTS_DIR = Path("../results")

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
results.to_csv(
    RESULTS_DIR / "member1_results.csv",
    index=False
)

print("Results saved successfully.")

In [ ]:
print("Saved model files:")

for file in MODEL_DIR.iterdir():
    print(" -", file.name)

print("\nSaved result files:")

for file in RESULTS_DIR.iterdir():
    print(" -", file.name)

In [ ]:
loaded_rf = joblib.load(
    MODEL_DIR / "random_forest.pkl"
)

test_predictions = loaded_rf.predict(X_test)

print("Loaded model prediction successful.")
print("First 5 predictions:")
print(test_predictions[:5])

In [ ]:
print("=" * 60)
print("MEMBER 1 - TRADITIONAL ML SUMMARY")
print("=" * 60)

for _, row in results.iterrows():
    print(f"\n{row['Model']}")
    print(f"MAE  : {row['MAE']:.4f}")
    print(f"RMSE : {row['RMSE']:.4f}")
    print(f"R²   : {row['R2']:.4f}")

print("\nModels saved in:")
print(MODEL_DIR)

print("\nResults saved in:")
print(RESULTS_DIR / "member1_results.csv")